In [18]:
import numpy as np
from itertools import product

from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Pauli

In [2]:
backend = AerSimulator()
sampler = Sampler(backend)

def w_state_circuit(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(1, n):
        qc.cx(0, i)
    # project to W using amplitude normalization
    return qc

n = 10
qc = w_state_circuit(n)

pm = generate_preset_pass_manager(backend=backend, optimization_level=2)
transpiled_circuit = pm.run(qc)
sampler.run([transpiled_circuit], shots=16384)


NameError: name 'AerSimulator' is not defined

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def w_state_circuit(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(1, n):
        qc.cx(0, i)
    # project to W using amplitude normalization
    return qc

n = 10
qc = w_state_circuit(n)
psi = Statevector.from_instruction(qc)
statevector = psi.data  # length 2^n
statevector

In [51]:
m = [
    np.array([[1,0],[0,0]], dtype=complex),
    np.array([[0,1],[0,0]], dtype=complex),
    np.array([[0,0],[1,0]], dtype=complex),
    np.array([[0,0],[0,1]], dtype=complex),
]


In [52]:
def kron_ops(op_list):
    out = op_list[0]
    for o in op_list[1:]:
        out = np.kron(out, o)
    return out

def generate_training_data(statevector, n, M):
    data = []
    for _ in range(M):
        indices = np.random.randint(0, 4, size=n)
        O = kron_ops([m[i] for i in indices])
        y = np.vdot(statevector, O @ statevector)
        data.append((indices, y))
    return data

training_data = generate_training_data(statevector, n, M=80)

In [57]:
import numpy as np
from itertools import product
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit import transpile

# --- Decomposition of single-qubit m_i into Pauli terms ---
# returns list[(coeff, 'I'/'X'/'Y'/'Z')]
def m_to_pauli_terms(i):
    if i == 0:   # |0><0| = (I+Z)/2
        return [(0.5+0j, 'I'), (0.5+0j, 'Z')]
    if i == 3:   # |1><1| = (I-Z)/2
        return [(0.5+0j, 'I'), (-0.5+0j, 'Z')]
    if i == 1:   # |0><1| = (X+iY)/2
        return [(0.5+0j, 'X'), (0.0+0.5j, 'Y')]
    if i == 2:   # |1><0| = (X-iY)/2
        return [(0.5+0j, 'X'), (0.0-0.5j, 'Y')]
    raise ValueError("i must be in {0,1,2,3}")

def expand_product_operator(m_indices):
    """
    Given m_indices (length n, values in 0..3), return list[(coeff, pauli_string)]
    where pauli_string is length n over 'I','X','Y','Z'
    """
    term_lists = [m_to_pauli_terms(i) for i in m_indices]
    expanded = []
    for choices in product(*term_lists):
        coeff = 1.0+0j
        paulis = []
        for c, p in choices:
            coeff *= c
            paulis.append(p)
        expanded.append((coeff, ''.join(paulis)))
    return expanded

# --- Measurement of a single Pauli string expectation value ---
def add_measurement_for_pauli_string(qc, pauli_string):
    """
    Adds basis rotations so that measuring in Z gives the Pauli-string eigenvalues.
    """
    for qubit, p in enumerate(pauli_string):
        if p == 'X':
            qc.h(qubit)
        elif p == 'Y':
            qc.sdg(qubit)
            qc.h(qubit)
        elif p == 'Z' or p == 'I':
            pass
        else:
            raise ValueError("Invalid Pauli character")
    qc.measure_all()

def pauli_string_expectation_from_counts(counts, pauli_string, n):
    """
    Compute <P> from bitstring counts after measuring in the appropriate basis.
    Qiskit returns bitstrings with qubit-0 on the RIGHT by default.
    """
    shots = sum(counts.values())
    exp = 0.0
    for bitstr, c in counts.items():
        # bitstr like '0101' with leftmost = highest classical bit
        # map to qubits: qubit q corresponds to bitstr[::-1][q]
        bits = bitstr[::-1]
        parity = 0
        for q, p in enumerate(pauli_string):
            if p == 'I':
                continue
            parity ^= int(bits[q])  # XOR bits on non-identity positions
        exp += (1 if parity == 0 else -1) * (c / shots)
    return exp


In [58]:
def estimate_expectation_Oj_from_measurements(prep_circuit, m_indices, shots=20000, backend=None):
    """
    Measurement-based estimator for y_j = <phi|Oj|phi>.
    Oj is defined via paper's m_i indices. Returns complex y_j.
    """
    n = prep_circuit.num_qubits
    if backend is None:
        backend = Aer.get_backend("qasm_simulator")

    expanded = expand_product_operator(m_indices)  # list[(coeff, pauli_string)]

    y = 0.0 + 0.0j
    for coeff, pstr in expanded:
        if pstr == 'I'*n:
            expval = 1.0  # always 1 for normalized state; measured version optional
        else:
            qc = QuantumCircuit(n)
            qc.compose(prep_circuit, inplace=True)
            add_measurement_for_pauli_string(qc, pstr)
            tqc = transpile(qc, backend)
            result = backend.run(tqc, shots=shots).result()
            counts = result.get_counts()
            expval = pauli_string_expectation_from_counts(counts, pstr, n)

        y += coeff * expval

    return y

In [ ]:
def generate_training_data_from_circuits(prep_circuit, n, M, shots=20000, seed=None):
    rng = np.random.default_rng(seed)
    data = []
    for _ in range(M):
        m_indices = rng.integers(0, 4, size=n).tolist()
        y = estimate_expectation_Oj_from_measurements(prep_circuit, m_indices, shots=shots)
        data.append((m_indices, y))
    return data


In [54]:
def mps_expectation(mps, op_indices):
    env = np.array([[1.0+0j]])

    for A, idx in zip(mps, op_indices):
        O = m[idx]
        Aconj = np.conj(A)

        tmp = np.einsum("ab, aic -> bic", env, A)
        tmp = np.einsum("bic, ij -> bjc", tmp, O)
        env = np.einsum("bjc, aic -> ba", tmp, Aconj)

    return env[0,0]


In [55]:
from scipy.optimize import minimize

def flatten_mps(mps):
    return np.concatenate([A.ravel() for A in mps])

def unflatten_mps(x, shapes):
    mps = []
    idx = 0
    for shape in shapes:
        size = np.prod(shape)
        mps.append(x[idx:idx+size].reshape(shape))
        idx += size
    return mps

def loss_fn(x, shapes, training_data):
    mps = unflatten_mps(x, shapes)
    loss = 0.0

    norm = mps_expectation(mps, [0]*len(mps))
    for indices, y in training_data:
        pred = mps_expectation(mps, indices) / norm
        loss += abs(pred - y)**2

    return loss / len(training_data)


In [56]:
# random initial MPS
D = 4
init_mps = [np.random.randn(1 if i==0 else D, 2, 1 if i==n-1 else D) + 1j*np.random.randn(1 if i==0 else D, 2, 1 if i==n-1 else D)
            for i in range(n)]

shapes = [A.shape for A in init_mps]
x0 = flatten_mps(init_mps)

res = minimize(
    loss_fn,
    x0,
    args=(shapes, training_data),
    method="BFGS",
    options={"gtol":1e-8}
)

trained_mps = unflatten_mps(res.x, shapes)

KeyboardInterrupt: 

In [49]:
import numpy as np

def mps_to_statevector(mps):
    """
    Convert an MPS into a full statevector.
    mps: list of tensors with shape (Dl, 2, Dr)
    """
    psi = mps[0]
    psi = psi.reshape(2, -1)  # (physical, bond)

    for A in mps[1:]:
        psi = np.tensordot(psi, A, axes=([1], [0]))
        # result shape: (phys_prev, phys_new, bond)
        psi = psi.reshape(-1, psi.shape[-1])

    return psi[:, 0]  # final bond dimension is 1

[array([[[ 2.15356245e-02+0.28122773j, -1.57396805e+00-3.05097979j,
           2.06406877e+00+0.91224185j,  1.55154105e-01+0.21767785j],
         [-1.12570603e-01+0.29173438j,  1.64815507e-01-1.86006643j,
           1.02547128e-01+0.70138816j, -1.14575534e-03-1.32491548j]]]),
 array([[[ 1.20200097-1.40800309j,  0.01165478+0.97010842j,
           0.1645313 +0.27650893j,  0.39778665+0.2469044j ],
         [-0.68272696+1.29532293j, -0.53865424+0.20311887j,
          -0.0270756 -1.10091371j, -0.68355049-0.42258354j]],
 
        [[ 1.86051834-0.84560645j,  0.0607497 +0.42061569j,
          -1.70441938-0.00579124j, -2.12324554+1.38146549j],
         [-0.86261359+0.60369629j,  0.38432575-0.52508413j,
           0.11912649-1.42234525j, -0.50387235-0.54084339j]],
 
        [[ 1.11937275-0.26330062j, -0.15813761+0.1540736j ,
          -1.69357348+0.71499201j,  1.06790057+2.45938582j],
         [-1.66665372-0.34336366j,  0.26475794+0.36116212j,
          -0.83149249+0.83017855j, -0.11361081-1.025

In [ ]:
def fidelity_state_mps(statevector, mps):
    """
    Fidelity between a normalized statevector |phi>
    and an (unnormalized) MPS |phi_tilde>.
    """
    psi_mps = mps_to_statevector(mps)

    overlap = np.vdot(statevector, psi_mps)
    norm_mps = np.vdot(psi_mps, psi_mps)

    return np.abs(overlap)**2 / np.real(norm_mps)

In [ ]:
# statevector from Qiskit
# trained_mps from optimization

F = fidelity_state_mps(statevector, trained_mps)
print("Fidelity:", F)